## **Naturaleza del servicio**
### *Retraso de propagación*
Utilizando unos auriculares un participante recibirá una secuencia de órdenes grabadas previamente. En cada simulación se utilizarán diversos retardos aleatorios según una distribución estadística (dentro de un rango). Otro participante, que está escuchando la grabación sin retardo evaluará la reacción del que las recibe a través de los auriculares con el retraso aleatorio.

**Objetivo**: rango máximo de retrasos admisible.

| Rango | 100-1900 ms (diff. varianza 150,300,450ms) | (4 rangos) |
|---|---|---|

Texto enunciado en el audio:

```
Este es un fragmento preparado para una de las pruebas del trabajo fin de máster "Sistema para la comunicación síncrona punto a multipunto: aplicación a una orquesta sinfónica distribuida espacialmente".

La prueba consistirá en una secuencia de órdenes que se irán enunciando. Si usted es el participante emisor, deberá prestar atención en el tiempo de reacción del otro participante. En caso contrario, si usted es el participante receptor, limítese a ejecutar las órdenes según vaya escuchándolas.

En cada rango definido se darán las órdenes con tres escenarios diferentes (de menos a más desfavorable). El objetivo para el emisor consistirá en determinar el rango máximo admisible.

- Asienta con la cabeza.
- Niegue con la cabeza.
- Levante la mano derecha.
- Levante la mano izquierda.
```

In [69]:
from lib.helper import play_audio
INTRO = "lib/Este es un fragmento.wav"
play_audio(INTRO, (1, -1))

Definición de la lista donde se encuentran los valores de la prueba. Cada una de las filas implica una simulación diferente que tiene un rango comprendido entre los dos valores (de izquierda a derecha).

Se ha calculado para cada uno de los puntos que representan el espacio de valores de la prueba su rango utilizando tres varianzas diferentes, ordenadas por tamaño (150ms, 300ms, 450ms).

In [70]:
import numpy as np
delay_set = np.ndarray(shape=(12,2), dtype=float)
dots = np.arange(0.1, 1.9, 0.45) # 100ms, 550ms, 1000ms, 1450ms
var_steps = [0.15, 0.3, 0.45] # 150ms, 300ms, 450ms

for i in range(len(dots)):
    value = np.stack([np.array([dots[i], dots[i] + step]) for step in var_steps])
    delay_set[i*len(var_steps):i*len(var_steps)+len(var_steps)] = value

delay_set

array([[0.1 , 0.25],
       [0.1 , 0.4 ],
       [0.1 , 0.55],
       [0.55, 0.7 ],
       [0.55, 0.85],
       [0.55, 1.  ],
       [1.  , 1.15],
       [1.  , 1.3 ],
       [1.  , 1.45],
       [1.45, 1.6 ],
       [1.45, 1.75],
       [1.45, 1.9 ]])

**NOTA**: aunque aquí se muestre una división por canales (L-R), cada señal se reproduce independientemente en unos auriculares, utilizando una configuración similar a la mostrada en la imagen.
<img src="lib/image.png" alt="drawing" width="900"/>

In [74]:
import time
from lib.helper import play_audio, StoppableThread

AUDIO_FILE = "lib/Instrucciones.wav"

print("> NASVP-01: from 100ms to 1900ms (4 ranges) with 150ms, 300ms, 450ms steps")

index = 0
for start, end in delay_set:
	delay = np.random.randint(start*1000, end*1000)
	DELAY = delay / 1000

	if index == 0:
		print("\t Range 1: 100ms - 550ms with 150ms step")
		play_audio("lib/Rango 1.wav", (1, 1))
	elif index == 3:
		print("\t Range 2: 550ms - 1000ms with 300ms step")
		play_audio("lib/Rango 2.wav", (1, -1))
	elif index == 6:
		print("\t Range 3: 1000ms - 1450ms with 450ms step")
		play_audio("lib/Rango 3.wav", (1, -1))
	elif index == 9:
		print("\t Range 4: 1450ms - 1900ms with 450ms step")
		play_audio("lib/Rango 4.wav", (1, -1))
	else:
		play_audio("lib/Cambio.wav", (1, 1))
	print("\t- Playing audio with {:.1f}ms delay ({:.1f}-{:.1f}ms)...".format(DELAY*1000, start*1000, end*1000), end="")
	a = StoppableThread(target=play_audio, args=(AUDIO_FILE,(1,-1),))
	b = StoppableThread(target=play_audio, args=(AUDIO_FILE,(-1,1),))

	a.start()
	time.sleep(DELAY)
	b.start()

	try:
		b.join()
		print(" Done!")
	except KeyboardInterrupt:
		a.kill()
		b.kill()
		print("\n\nEarly stopping...")
		break

	index += 1

> NASVP-01: from 100ms to 1900ms (4 ranges) with 150ms, 300ms, 450ms steps
	 Range 1: 100ms - 550ms with 150ms step
	- Playing audio with 242.0ms delay (100.0-250.0ms)... Done!
	- Playing audio with 130.0ms delay (100.0-400.0ms)... Done!
	- Playing audio with 525.0ms delay (100.0-550.0ms)... Done!
	 Range 2: 550ms - 1000ms with 300ms step
	- Playing audio with 643.0ms delay (550.0-700.0ms)... Done!
	- Playing audio with 566.0ms delay (550.0-850.0ms)... Done!
	- Playing audio with 960.0ms delay (550.0-1000.0ms)... Done!
	 Range 3: 1000ms - 1450ms with 450ms step
	- Playing audio with 1013.0ms delay (1000.0-1150.0ms)... Done!
	- Playing audio with 1273.0ms delay (1000.0-1300.0ms)...

Early stopping...
